# LeaseGuard Colab orchestrator

This notebook is the only place that should download source documents, professional benchmarks, or run official evaluators. The local laptop keeps code, schemas, manifests, and tiny fixtures.

Set `TASK` to one of:

- `process-documents` — parse approved sources into intermediate records
- `build-dataset` — assemble Dataset v1 from approved examples
- `regression` — internal lease product checks
- `evaluate-legalbench`, `evaluate-cuad`, `evaluate-contractnli`, `evaluate-legalbench-rag`
- `baseline-qwen35-4b`, `baseline-qwen35-9b`, `baseline-gemma3-12b` — unmodified base-model product-task matrix
- `compare-baselines` — select the strongest practical T4 candidate from saved runs

Dedicated Phase 6 notebooks also exist:

- `colab_baseline_qwen35_4b.ipynb`
- `colab_baseline_qwen35_9b.ipynb`
- `colab_baseline_gemma3_12b.ipynb`
- `colab_baseline_compare.ipynb`

Mount Google Drive first so raw files, processed records, checkpoints, reports, and dataset exports survive session limits.


In [ ]:
import os
from pathlib import Path

TASK = "baseline-qwen35-4b"
REPO_URL = "https://github.com/YOUR_USER/LeaseGuard.git"
REPO_DIR = Path("/content/LeaseGuard")
DRIVE_ROOT = Path("/content/drive/MyDrive/leaseguard")
RAW_ROOT = DRIVE_ROOT / "raw"
PROCESSED_ROOT = DRIVE_ROOT / "processed"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
REPORT_DIR = DRIVE_ROOT / "reports"
CHECKOUT_ROOT = DRIVE_ROOT / "evaluators"
DATASET_ROOT = DRIVE_ROOT / "datasets"
EXAMPLES_ROOT = DRIVE_ROOT / "dataset_examples"
BASELINE_REPORT_DIR = REPORT_DIR / "baselines"
BASELINE_CHECKPOINT_DIR = CHECKPOINT_DIR / "baselines"
HF_TOKEN = ""

print({"task": TASK, "repo": str(REPO_DIR), "drive": str(DRIVE_ROOT)})

In [ ]:
import sys
from subprocess import check_call

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; using local paths already set above.")

if not REPO_DIR.exists():
    check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
os.chdir(REPO_DIR)
check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
if TASK.startswith("baseline-") or TASK.startswith("evaluate-"):
    check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers",
            "accelerate",
            "bitsandbytes",
            "huggingface_hub",
        ]
    )
for path in (
    RAW_ROOT,
    PROCESSED_ROOT,
    CHECKPOINT_DIR,
    REPORT_DIR,
    CHECKOUT_ROOT,
    DATASET_ROOT,
    EXAMPLES_ROOT,
    BASELINE_REPORT_DIR,
    BASELINE_CHECKPOINT_DIR,
):
    path.mkdir(parents=True, exist_ok=True)

In [ ]:
from leaseguard.dataset.cli import main as dataset_main
from leaseguard.evaluation.cli import main as evaluation_main
from leaseguard.ingestion.cli import main as ingestion_main

if TASK == "regression":
    raise SystemExit(
        evaluation_main(["regression", "--output", str(REPORT_DIR / "regression.json")])
    )

if TASK == "process-documents":
    os.environ["LEASEGUARD_ALLOW_DOCUMENT_DOWNLOAD"] = "1"
    raise SystemExit(
        ingestion_main(
            [
                "process",
                "--raw-root",
                str(RAW_ROOT),
                "--processed-root",
                str(PROCESSED_ROOT),
                "--checkpoint-dir",
                str(CHECKPOINT_DIR),
                "--search-root",
                str(REPO_DIR),
                "--output",
                str(REPORT_DIR / "pipeline.json"),
                "--allow-download",
            ]
        )
    )

if TASK == "build-dataset":
    examples = (
        EXAMPLES_ROOT
        if any(EXAMPLES_ROOT.rglob("*.json"))
        else REPO_DIR / "data" / "samples" / "dataset_v1" / "examples"
    )
    raise SystemExit(
        dataset_main(
            [
                "build",
                "--examples",
                str(examples),
                "--quality-review",
                str(REPO_DIR / "data" / "samples" / "dataset_v1" / "quality_review.json"),
                "--output",
                str(DATASET_ROOT / "dataset.v1.json"),
            ]
        )
    )

if TASK == "compare-baselines":
    runs = sorted(BASELINE_REPORT_DIR.glob("*-schema-full-4bit.json"))
    raise SystemExit(
        evaluation_main(
            [
                "compare-baselines",
                *[str(path) for path in runs],
                "--output",
                str(BASELINE_REPORT_DIR / "phase6_comparison.json"),
            ]
        )
    )

if TASK.startswith("baseline-"):
    os.environ["LEASEGUARD_ALLOW_MODEL_DOWNLOAD"] = "1"
    candidate = TASK.removeprefix("baseline-")
    from leaseguard.evaluation.baseline import get_candidate, load_baseline_registry

    registry = load_baseline_registry()
    spec = get_candidate(registry, candidate)
    if spec.gated:
        if not HF_TOKEN:
            raise SystemExit("Set HF_TOKEN for gated models such as Gemma.")
        from huggingface_hub import login

        login(token=HF_TOKEN)
    examples = (
        EXAMPLES_ROOT
        if any(EXAMPLES_ROOT.rglob("*.json"))
        else REPO_DIR / "data" / "samples" / "dataset_v1" / "examples"
    )
    comparisons = [
        item.comparison_id for item in registry.comparisons if candidate in item.candidate_ids
    ]
    status = 0
    for comparison_id in comparisons:
        status = evaluation_main(
            [
                "baseline",
                "--candidate",
                candidate,
                "--comparison",
                comparison_id,
                "--backend",
                "transformers",
                "--allow-download",
                "--examples",
                str(examples),
                "--output-dir",
                str(BASELINE_REPORT_DIR),
                "--checkpoint-dir",
                str(BASELINE_CHECKPOINT_DIR),
                "--hardware",
                "Google Colab T4",
            ]
        )
        print(comparison_id, "exit", status)
    raise SystemExit(status)

benchmark_id = TASK.removeprefix("evaluate-")
os.environ["LEASEGUARD_ALLOW_BENCHMARK_DOWNLOAD"] = "1"
prepare_status = evaluation_main(
    [
        "prepare-official",
        "--benchmark",
        benchmark_id,
        "--checkout-root",
        str(CHECKOUT_ROOT),
        "--allow-download",
    ]
)
print("prepare exit", prepare_status)
raise SystemExit(
    evaluation_main(
        [
            "official-status",
            "--benchmark",
            benchmark_id,
            "--checkout-root",
            str(CHECKOUT_ROOT),
        ]
    )
)